# Fake Data Tests for Wiener-SVD Unfolding

- Only use MC stat and GENIE syst -- testing robustness of interaction uncertainties

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl
from os import path
import sys
from tqdm import tqdm
import datetime

# local imports
sys.path.append('../../')
from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.numucc_1p0pi.selection_definitions import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.makedf.util import *
from pyanalib.split_df_helpers import *
from pyanalib.stat_helpers import *
from pyanalib.pandas_helpers import *
from pyanalib.variable_calculator import get_cc1p0pi_tki
from pyanalib.pandas_helpers import pad_column_name
from makedf.constants import *

plt.style.use("presentation.mplstyle")
# plt.style.use('seaborn-v0_8-colorblind')
# mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=sns.color_palette("hls", 8))
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=plt.get_cmap('Set1').colors)

import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)

In [ ]:
save_fig = False
today = datetime.datetime.now().strftime("%Y%m%d")
save_fig_dir = "/exp/sbnd/data/users/munjung/plots/numucc1p0pi/unfolding-fake_data_tests-{}".format(today)

# load dataframes

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"

## -- MC 
# mc_file = path.join(file_dir, "MC", "BNB_cosmics", "genie_wgts-CCQE", "aa_geniewgts_CCQE.df")
mc_file = path.join(file_dir, "MC", "BNB_cosmics", "aa_sel_mup-geniewgts.df")
mc_split_df = pd.read_hdf(mc_file, key="split")
mc_n_split = get_n_split(mc_file)
print("mc_n_split: %d" %(mc_n_split))
print_keys(mc_file)

n_max_concat = 3
mc_keys2load = ['hdr', "mcnu", 'evt'] 
mc_dfs = load_dfs(mc_file, mc_keys2load, n_max_concat=n_max_concat)
mc_hdr_df = mc_dfs['hdr']
mc_nu_df = mc_dfs['mcnu']
mc_evt_df = mc_dfs['evt']

## ==== POT
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
# target_pot = 1e20
target_pot = mc_tot_pot
mc_pot_scale = target_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))

mc_evt_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_evt_df))
mc_nu_df["pot_weight"] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
# add multiindex column index "mc" so that branch names match evt_df
# TODO: fix so I don't have to do this
mc_nu_df.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in mc_nu_df.columns])     # match # of column levels

In [ ]:
print("==== selected events ====")
mc_evt_df.loc[:,'nuint_categ'] = get_int_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)
print(mc_evt_df.nuint_categ.value_counts())
print(mc_evt_df.genie_categ.value_counts())

print("==== all events ====")
mc_nu_df.loc[:,'nuint_categ'] = get_int_category(mc_nu_df)
mc_nu_df.loc[:,'genie_categ'] = get_genie_category(mc_nu_df)
print(mc_nu_df.nuint_categ.value_counts())
print(mc_nu_df.genie_categ.value_counts())


# Constants

In [ ]:
import uproot

# TODO: z-dependence?
# flux file, units: /m^2/10^6 POT 
# 50 MeV bins
fluxfile = "/exp/sbnd/data/users/munjung/flux/sbnd_original_flux.root"
flux = uproot.open(fluxfile)
print(flux.keys())
numu_flux = flux["flux_sbnd_numu"].to_numpy()
bin_edges = numu_flux[1]
flux_vals = numu_flux[0]

fig, ax = plt.subplots()
plt.hist(bin_edges[:-1], bins=bin_edges, weights=flux_vals, histtype="step", linewidth=2, color="C0")
plt.xlim(0, 3)
plt.xlabel("Neutrino Energy [GeV]")
plt.ylabel("Flux [/m$^{2}$/10$^{6}$ POT]")
plt.title("SBND $\\nu_\\mu$ Flux")


# get integrated flux
integrated_flux = flux_vals.sum()
integrated_flux /= 1e4 # to cm2
INTEGRATED_FLUX = integrated_flux * mc_tot_pot / 1e6 # POT
print("Integrated flux: %.3e" % INTEGRATED_FLUX)

V_SBND = 380 * 380 * 440 # cm3, the active volume of the detector 
NTARGETS = RHO * V_SBND * N_A / M_AR
print("# of targets: ", NTARGETS)

# TODO: fix scalar overflow error
XSEC_UNIT = 1 / (INTEGRATED_FLUX * NTARGETS)
print("xsec unit: ", XSEC_UNIT)
if XSEC_UNIT == 0:
    XSEC_UNIT = 1e-38

# Choose variable

In [ ]:
var_config = VariableConfig.muon_momentum()

# Systematic Uncertainties

In [ ]:
file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_09"
mcstat_syst = np.load(file_dir + "/mcstat_syst_dict.npz")
flux_syst = np.load(file_dir + "/flux_syst_dict.npz")
g4_syst = np.load(file_dir + "/g4_syst_dict.npz")
cosmics_syst = np.load(file_dir + "/cosmics_syst_dict.npz")
genie_syst = np.load(file_dir + "/genie_syst_dict.npz")

file_dir = "/exp/sbnd/data/users/munjung/xsec/2025Spring_v10_06_00_10"
detvar_syst = np.load(file_dir + "/detvar_syst_dict.npz")

In [ ]:
fig, ax = plt.subplots()

frac_uncert_total = np.zeros(len(var_config.bin_centers))

systs      = [mcstat_syst, genie_syst, flux_syst, g4_syst, cosmics_syst, detvar_syst]
syst_names = ["MC stat.", "GENIE", "Flux", "G4", "Cosmics", "Detector"]

for syst_name, syst in zip(syst_names, systs):
    syst_uncert = np.sqrt(np.diag(syst[var_config.var_save_name]))
    frac_uncert_total += syst_uncert ** 2
    plt.hist(var_config.bin_centers, bins=var_config.bins, weights=syst_uncert * 1e2,   histtype="step", linewidth=2, label=syst_name)

frac_uncert_total = np.sqrt(frac_uncert_total)
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total * 1e2,    histtype="step", linewidth=2, color="k",  label="Total")

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.ylim(0, max(frac_uncert_total*1e2) * 1.4)

plt.xlabel(var_config.var_labels[1])
plt.ylabel("Uncertainty [%]")
plt.legend(fontsize=11, ncol=3, loc="upper center")

plt.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
plt.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
plt.minorticks_on()

if save_fig:
    plt.savefig("{}/uncertainty_breakdown-{}.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show();

# Distributions

Draw true (before event selection) and reco (after event selection) muon momentum distributions of signal events.
Print entries for double check.

In [ ]:
ret_dists = get_distributions(mc_evt_df, mc_nu_df, var_config)

In [ ]:
nevts_signal_truth, _, _     = plt.hist(ret_dists["var_truth_signal"],     bins=var_config.bins, weights=ret_dists["weight_truth_signal"], histtype="step", label="All Signal in MC")
nevts_signal_sel_truth, _, _ = plt.hist(ret_dists["var_signal_sel_truth"], bins=var_config.bins, weights=ret_dists["weight_signal"],       histtype="step", label="Selected Signal, True")
nevts_signal_sel_reco, _, _  = plt.hist(ret_dists["var_signal_sel_reco"],  bins=var_config.bins, weights=ret_dists["weight_signal"],       histtype="step", label="Selected Signal, Reco", color="k")

plt.legend()
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events / Bin")
plt.xlim(var_config.bins[0], var_config.bins[-1])

if save_fig:
    plt.savefig("{}/{}-sel_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show();

# Unfolding

In [ ]:
# --- config for Wiener-SVD unfolding ---
C_type = 2
Norm_type = 0.5

## Closure test (Asimov data)
- use MC signal as fake data

In [ ]:
var_total = ret_dists["var_total_mc"]
weights_total = ret_dists["weights_total_mc"]
var_categ = ret_dists["var_per_nuint_categ_mc"]
weights_categ = ret_dists["weights_per_categ"]

bins = var_config.bins
plot_labels = [var_config.var_labels[1], "Events / Bin"]
colors = topology_colors
labels = topology_labels
save_name = "{}/{}-closure_test_topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name)
fake_data, background_cv = plot_topology_breakdown(var_categ, weights_categ, var_total, weights_total, 
                            bins,
                            plot_labels, 
                            colors=colors, labels=labels, 
                            save_fig=save_fig, save_name=save_name)

var_categ = ret_dists["var_per_genie_mode_mc"]
weights_categ = ret_dists["weights_per_genie_mode"]
colors = genie_mode_colors
labels = genie_mode_labels
save_name = "{}/{}-closure_test_genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name)
plot_genie_breakdown(var_categ, weights_categ, var_total, weights_total, 
                     bins,
                     plot_labels, 
                     colors=colors, labels=labels,
                     save_fig=save_fig, save_name=save_name)

In [ ]:
save_fig_name = "{}/{}-reco_vs_true".format(save_fig_dir, var_config.var_save_name)
reco_vs_true = get_smear_matrix(ret_dists["var_signal_sel_truth"], ret_dists["var_signal_sel_reco"], 
                                var_config.bins, var_labels=var_config.var_labels,
                                weights=mc_evt_df[mc_evt_df.nuint_categ == 1]["pot_weight"],
                                save_fig=save_fig, save_fig_name=save_fig_name)

eff = get_eff(reco_vs_true, nevts_signal_truth)

save_fig_name = "{}/{}-response_matrix".format(save_fig_dir, var_config.var_save_name)
Response = get_response_matrix(reco_vs_true, eff, 
                               var_config.bins, var_labels=var_config.var_labels,
                               save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
Measured = nevts_signal_sel_reco * XSEC_UNIT 
Model = nevts_signal_truth * XSEC_UNIT
Covariance = cov_from_fraccov(mcstat_syst[var_config.var_save_name], nevts_signal_sel_reco) * XSEC_UNIT**2
unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)
print(unfold.keys())
# decomp_cov = Matrix_Decomp(Model, unfold['SystUnfoldCov'])

In [ ]:
models = [Model]
model_names = ["SBND Baseline Model"]

plot_labels = [var_config.var_labels[0], var_config.xsec_label]
save_name = "{}-closure_test_output.pdf".format(var_config.var_save_name)
plot_unfolded_result(unfold, var_config.bins, Measured, models, 
                      plot_labels, model_names, 
                      save_fig=save_fig, save_name=save_name,
                      closure_test=True)

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, "closure_test")
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], var_config, 
             "",
             save_fig=save_fig, save_fig_name=save_fig_name)

## Fake Data Tests

- use alternate MC as fake data

In [ ]:
# use only stat unc and xsec unc for fake data tests
covariance_frac = genie_syst[var_config.var_save_name] + mcstat_syst[var_config.var_save_name]
covariance = cov_from_fraccov(covariance_frac, nevts_signal_sel_reco) * XSEC_UNIT**2

In [ ]:
unfolded_plot_labels = [var_config.var_labels[0], var_config.xsec_label]
smearmat_plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]

In [ ]:
class FakeDataWeights:
    def __init__(self, mc_evt_df, mc_nu_df, var_config):
        self.mc_evt_df = mc_evt_df
        self.mc_nu_df = mc_nu_df
        self.var_config = var_config

    def get_weights(self, test_name, **kwargs):
        # Always start from ones (unless otherwise needed)
        # TODO: place normalization here?
        weights_fake_data = np.ones(len(self.mc_evt_df))
        weight_fakedata_signal_truth = np.ones(len(self.mc_nu_df[self.mc_nu_df.nuint_categ == 1]))
        
        # MEC normalization
        if test_name == "mec_test":
            scale_factor = kwargs.get("scale_factor", 0.5)
            weights_fake_data[self.mc_evt_df.mc.genie_mode == 10] *= scale_factor
            weight_fakedata_signal_truth[self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].mc.genie_mode == 10] *= scale_factor

        # QE normalization
        elif test_name == "qe_test":
            scale_factor = kwargs.get("scale_factor", 1.2)
            weights_fake_data[self.mc_evt_df.mc.genie_mode == 0] *= scale_factor
            weight_fakedata_signal_truth[self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].mc.genie_mode == 0] *= scale_factor

        # np normalization
        elif test_name == "np_test":
            Np_scale = kwargs.get("Np_scale", 2)
            weights_fake_data[self.mc_evt_df.nuint_categ == 2] *= Np_scale
            weight_fakedata_signal_truth[self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].nuint_categ == 2] *= Np_scale

        # sig normalization
        elif test_name == "sig_test":
            sig_scale = kwargs.get("sig_scale", 1.2)
            weights_fake_data[self.mc_evt_df.nuint_categ == 1] *= sig_scale
            weight_fakedata_signal_truth[self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].nuint_categ == 1] *= sig_scale

        # Q2 tilt
        elif test_name.startswith("q2_test_alpha_"):
            alpha = kwargs.get("alpha", 0.3)
            Q2 = 2 * self.mc_evt_df.mc.E * self.mc_evt_df.mu.pfp.trk.truth.p.startE * (1 - self.mc_evt_df.mu.pfp.trk.truth.p.dir.z)
            weights_fake_data *= np.ones(len(self.mc_evt_df)) + alpha * (Q2 - Q2.mean())/Q2.mean()
            weights_fake_data[np.isnan(weights_fake_data)] = 1
            assert np.isnan(weights_fake_data).sum() == 0
            Q2_nu = self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].mc.Q2
            weight_fakedata_signal_truth *= np.ones(len(Q2_nu)) + alpha * (Q2_nu - Q2_nu.mean())/Q2_nu.mean()
        
        # cos(theta) scale
        elif test_name.startswith("costh_weight_scale_"):
            scale_factor = kwargs.get("scale_factor", 0.7)
            weights_fake_data[self.mc_evt_df.mu.pfp.trk.truth.p.dir.z > 0.9] *= scale_factor
            weight_fakedata_signal_truth[self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].mc.mu.dir.z > 0.9] *= scale_factor

        # Proton P tilt
        elif test_name.startswith("proton_P_tilt_alpha_"):
            alpha = kwargs.get("alpha", 0.3)
            P_p_evt = self.mc_evt_df.p.pfp.trk.truth.p.totp
            weights_fake_data = np.ones(len(self.mc_evt_df)) + alpha * (P_p_evt - P_p_evt.mean())/P_p_evt.mean()
            P_p_nu = self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].mc.p.totp
            weight_fakedata_signal_truth = np.ones(len(P_p_nu)) + alpha * (P_p_nu - P_p_nu.mean())/P_p_nu.mean()

        # Bump
        elif test_name.startswith("bump_"):
            bump_pos = kwargs.get("bump_pos", 0.6)
            bump_width = kwargs.get("bump_width", 0.0015)
            bump_height = kwargs.get("bump_height", 0.001)
            bump_height = bump_height*len(self.mc_evt_df)/len(self.var_config.bin_centers)
            bump_var_evt = self.mc_evt_df[self.var_config.var_evt_truth_col]
            weights_fake_data = np.ones(len(self.mc_evt_df)) + bump_height * np.exp(-0.5 * (bump_var_evt - bump_pos)**2 / bump_width**2)
            weights_fake_data[np.isnan(weights_fake_data)] = 1.
            bump_var_nu = self.mc_nu_df[self.mc_nu_df.nuint_categ == 1][self.var_config.var_nu_col]
            weight_fakedata_signal_truth = np.ones(len(bump_var_nu)) + bump_height * np.exp(-0.5 * (bump_var_nu - bump_pos)**2 / bump_width**2)
        
        # TODO
        # # Enhance tail
        # elif test_name.startswith("enhance_tail_alpha_"):
        #     alpha = kwargs.get("alpha", 0.7)
        #     weights_fake_data = np.ones(len(self.mc_evt_df))
        #     weights_fake_data[self.mc_evt_df.del_p > 0.25] = alpha
        #     weights_fake_data[np.isnan(weights_fake_data)] = 1.
        #     weight_fakedata_signal_truth = np.ones(len(self.mc_nu_df[self.mc_nu_df.nuint_categ == 1]))
        #     weight_fakedata_signal_truth[self.mc_nu_df[self.mc_nu_df.nuint_categ == 1].mc.del_p > 0.25] = alpha
        #     weight_fakedata_signal_truth[np.isnan(weight_fakedata_signal_truth)] = 1.

        else:
            raise ValueError(f"Unknown test_name '{test_name}' provided.")

        return weights_fake_data, weight_fakedata_signal_truth

In [ ]:
# TODO: calculate this when making dfs
mc_evt_df[("mu","pfp","trk","truth","p","dir","z")] = mc_evt_df.mu.pfp.trk.truth.p.genp.z / mc_evt_df.mu.pfp.trk.truth.p.totp

In [ ]:
fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)

test_name = "mec_test"
scale_factor = 0.5
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=scale_factor)

test_name = "qe_test"
scale_factor = 1.2
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=scale_factor)

test_name = "np_test"
Np_scale = 2
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, Np_scale=Np_scale)

test_name = "sig_test"
sig_scale = 1.2
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, sig_scale=sig_scale)

test_name = "q2_test_alpha_0.3"
alpha = 0.3
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, alpha=alpha)

test_name = "costh_weight_scale_0.7"
scale_factor = 0.7
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, scale_factor=scale_factor)

test_name = "proton_P_tilt_alpha_0.3"
alpha = 0.3
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, alpha=alpha)

test_name = "bump_0.6_0.0015_0.001"
bump_pos = 0.6
bump_width = 0.0015
bump_height = 0.001
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, bump_pos=bump_pos, bump_width=bump_width, bump_height=bump_height)

In [ ]:
nevts_fakedata_reco, _ = np.histogram(ret_dists["var_total_mc"], bins=var_config.bins, weights=weights_fake_data)
nevts_nomdata_signal_truth, _ = np.histogram(ret_dists["var_truth_signal"], bins=var_config.bins)
nevts_fakedata_signal_truth, _ = np.histogram(ret_dists["var_truth_signal"], bins=var_config.bins, weights=weight_fakedata_signal_truth)

fig, ax = plt.subplots()
plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_reco, histtype="step", label="all selected events")
plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_nomdata_signal_truth, histtype="step", label="nominal MC, all signal")
plt.hist(var_config.bin_centers, var_config.bins, weights=nevts_fakedata_signal_truth, histtype="step", label="alternative MC, all signal")
plt.legend()
plt.show();

In [ ]:
var_total = ret_dists["var_total_mc"]
weights_total = weights_fake_data

var_categ = ret_dists["var_per_nuint_categ_mc"]
weights_categ = ret_dists["weights_per_categ"]

plot_labels = [var_config.var_labels[1], "Events / Bin"]
colors = topology_colors
labels = topology_labels
save_name = "{}/{}-{}-topology_breakdown.pdf".format(save_fig_dir, test_name, var_config.var_save_name)
fake_data, background_cv = plot_topology_breakdown(var_categ, weights_categ, 
                            var_total, weights_total, 
                            var_config.bins,
                            plot_labels, 
                            colors=colors, labels=labels, 
                            save_fig=save_fig, save_name=save_name)

var_categ = ret_dists["var_per_genie_mode_mc"]
weights_categ = ret_dists["weights_per_genie_mode"]
colors = genie_mode_colors
labels = genie_mode_labels
save_name = "{}/{}-{}-genie_mode_breakdown.pdf".format(save_fig_dir, test_name, var_config.var_save_name)
plot_genie_breakdown(var_categ, weights_categ, 
                     var_total, weights_total, 
                     var_config.bins,
                     plot_labels, 
                     colors=colors, labels=labels,
                     save_fig=save_fig, save_name=save_name)

In [ ]:
measured = (fake_data - background_cv) * XSEC_UNIT
model = nevts_nomdata_signal_truth * XSEC_UNIT
unfold = WienerSVD(Response, model, measured, covariance, C_type, Norm_type)

In [ ]:
models = [model, nevts_fakedata_signal_truth * XSEC_UNIT]
model_names = ["SBND Baseline Model", "Fake Data"]

save_name = "{}/{}-{}-unfolded_event_rates.pdf".format(save_fig_dir, test_name, var_config.var_save_name)
plot_unfolded_result(unfold, var_config.bins, measured, models, 
                     unfolded_plot_labels, model_names, save_fig=save_fig, save_name=save_name)

save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, test_name, var_config.var_save_name)
plot_heatmap(unfold["AddSmear"], var_config, 
             "",
             save_fig=save_fig, save_fig_name=save_fig_name)

# Check distributions for fake data

## $Q^2$ reweight

In [ ]:
fake_weight_obj = FakeDataWeights(mc_evt_df, mc_nu_df, var_config)
Q2 = 2 * mc_evt_df.mc.E * mc_evt_df.mu.pfp.trk.truth.p.startE * (1 - mc_evt_df.mu.pfp.trk.truth.p.dir.z)
test_name = "q2_test_alpha_0.3"
alpha = 0.3
weights_fake_data, weight_fakedata_signal_truth = fake_weight_obj.get_weights(test_name, alpha=alpha)

plt.hist(Q2, bins=bins, histtype="step", color="black", label="Nominal")
plt.hist(Q2, bins=bins, weights=weights_fake_data, histtype="step", color="red", label="Reweighted, $\\alpha=${}".format(alpha))
plt.xlabel("$Q^2$")
plt.ylabel("Events / Bin")
plt.legend()
if save_fig:
    plt.savefig("{}/Q2_reweight_{}.pdf".format(save_fig_dir, alpha), bbox_inches="tight")
plt.show();